# Font Settings by Cell Type (`window.craftFonts`)

> **Feature 4.1:** Per-cell-type font family, size, line height, and weight customization for **Prompt**, **Assistant response**, **Note**, **Code**, and **Raw** cells with live-rendered preview cards, instant live-reload on the active dialog, Monaco editor syncing, `localStorage` persistence, and **Copy as CRAFT.css snippet**.

## Templates (`#| export_ft`)

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = {
    'font_modal': craft_overlay(
        craft_card(
            Div(
                Strong('Font Settings by Cell Type'),
                Button('×', type='button', cls='craft-modal-x', aria_label='Close', data_ref='close'),
                cls='craft-modal-head',
            ),
            Div(
                Div(cls='craft-font-tabs', role='tablist', data_ref='tabs'),
                Div(cls='craft-font-body', data_ref='body'),
                cls='craft-font-main',
            ),
            Div(
                craft_btn('Copy as CRAFT.css snippet', size='xs', data_ref='copyCss'),
                craft_btn('Reset to defaults', size='xs', data_ref='reset'),
                craft_btn('Save & Close', variant='primary', size='xs', data_ref='save'),
                cls='craft-modal-foot',
            ),
            cls='craft-font-card',
        ),
        id='craft-font-modal', role='dialog', aria_modal='true', aria_label='Font Settings by Cell Type',
    ),
}

## Typography Defaults, Persistence & CSS Application (`#| export_js_iife`)


In [ ]:
#| export_js_iife CRAFT.js craftFonts
#| eval: false
window.craftFonts?.disable?.();

const CORE = window.__craftCMCore;
if (!CORE) return;
const {el, on, store, mountModal, notify} = CORE;
const ac = new AbortController(), sig = {signal: ac.signal};
const STORE_KEY = 'craft-font-settings-v1';

const TYPES = [
  {key: 'prompt',   label: 'Prompt',             defFamily: "'Craft Encode Sans', -apple-system, sans-serif", defSize: 11, defLh: 1.2, defWeight: 360, sample: 'You: Summarize the trade-offs of this indexing strategy in 3 bullet points.'},
  {key: 'response', label: 'Assistant response',  defFamily: "'Craft Encode Sans', -apple-system, sans-serif", defSize: 11, defLh: 1.25, defWeight: 360, sample: 'Assistant: Covering indexes avoid heap lookups, at the cost of extra write amplification and storage.'},
  {key: 'note',     label: 'Note',               defFamily: "'Craft Encode Sans', -apple-system, sans-serif", defSize: 11, defLh: 1.25, defWeight: 360, sample: 'Design note — keep per-cell typography variables falling back to --craft-prose and --craft-code.'},
  {key: 'code',     label: 'Code',               defFamily: "'Craft Iosevka', monospace",                     defSize: 10, defLh: 1.2,  defWeight: 400, sample: 'def fib(n: int) -> int:\n    return n if n < 2 else fib(n - 1) + fib(n - 2)'},
  {key: 'raw',      label: 'Raw',                defFamily: "'Craft Iosevka', monospace",                     defSize: 10, defLh: 1.25, defWeight: 400, sample: '{"status": "ok", "tokens": 1257, "mode": "raw"}'},
];

const FAMILY_PRESETS = [
  ["Craft Encode Sans (default prose)", "'Craft Encode Sans', -apple-system, sans-serif"],
  ["Craft Iosevka (default code)", "'Craft Iosevka', monospace"],
  ["System Sans-Serif", "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif"],
  ["System Serif", "Georgia, 'Iowan Old Style', 'Times New Roman', serif"],
  ["System Monospace", "ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace"],
];

const defaults = () => Object.fromEntries(TYPES.map(t => [t.key, {
  family: t.defFamily, size: t.defSize, lh: t.defLh, weight: t.defWeight,
}]));

const loadSettings = () => {
  const saved = store.json(STORE_KEY, null);
  const base = defaults();
  if (!saved || typeof saved !== 'object') return base;
  for (const t of TYPES) {
    if (saved[t.key]) base[t.key] = {...base[t.key], ...saved[t.key]};
  }
  return base;
};

let current = loadSettings();
let modal = null, activeTab = 'prompt';

function syncMonaco(codeCfg) {
  const mon = window.monaco?.editor;
  if (!mon || !codeCfg) return;
  const fs = parseFloat(codeCfg.size) || 10;
  const lh = Math.round(fs * (parseFloat(codeCfg.lh) || 1.2));
  const fam = codeCfg.family || 'monospace';
  try {
    mon.getEditors?.().forEach(ed => ed.updateOptions({fontSize: fs, lineHeight: lh, fontFamily: fam}));
    mon.getDiffEditors?.().forEach(ed => ed.updateOptions({fontSize: fs, lineHeight: lh, fontFamily: fam}));
    mon.remeasureFonts?.();
  } catch {}
}

function applySettings(cfg = current) {
  const root = document.documentElement.style;
  for (const t of TYPES) {
    const c = cfg[t.key];
    if (!c) continue;
    root.setProperty(`--craft-font-${t.key}-family`, c.family);
    root.setProperty(`--craft-font-${t.key}-size`, `${c.size}px`);
    root.setProperty(`--craft-font-${t.key}-lh`, String(c.lh));
    root.setProperty(`--craft-font-${t.key}-weight`, String(c.weight));
  }
  if (cfg.code) {
    root.setProperty('--craft-code', `${cfg.code.size}px`);
    root.setProperty('--craft-code-lh', String(cfg.code.lh));
    root.setProperty('--craft-code-family', cfg.code.family);
    syncMonaco(cfg.code);
  }
}

function buildCssSnippet(cfg = current) {
  const lines = [':root {'];
  for (const t of TYPES) {
    const c = cfg[t.key];
    lines.push(`  --craft-font-${t.key}-family: ${c.family};`);
    lines.push(`  --craft-font-${t.key}-size: ${c.size}px;`);
    lines.push(`  --craft-font-${t.key}-lh: ${c.lh};`);
    lines.push(`  --craft-font-${t.key}-weight: ${c.weight};`);
  }
  lines.push('}');
  return lines.join('\n');
}


## Font Settings Modal, Hotkeys & Public API (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftFonts
#| eval: false
function renderModal() {
  if (!modal) return;
  modal.tabs.replaceChildren(...TYPES.map(t => el('button', {
    type: 'button', role: 'tab', 'aria-selected': String(t.key === activeTab),
    cls: `craft-font-tab${t.key === activeTab ? ' is-active' : ''}`,
    onClick: () => { activeTab = t.key; renderModal(); },
  }, t.label)));

  const t = TYPES.find(x => x.key === activeTab) || TYPES[0];
  const cfg = current[t.key];
  const preview = el('div', {
    cls: `craft-font-preview is-${t.key}`,
    style: {fontFamily: cfg.family, fontSize: `${cfg.size}px`, lineHeight: String(cfg.lh), fontWeight: String(cfg.weight)},
  }, t.sample);

  const updatePreview = () => {
    Object.assign(preview.style, {
      fontFamily: cfg.family, fontSize: `${cfg.size}px`, lineHeight: String(cfg.lh), fontWeight: String(cfg.weight),
    });
    applySettings(current);
  };

  const famSel = el('select', {cls: 'uk-select craft-font-ctrl', aria_label: 'Font family preset'},
    ...FAMILY_PRESETS.map(([lbl, val]) => el('option', {value: val, selected: cfg.family === val}, lbl)),
    el('option', {value: '__custom__', selected: !FAMILY_PRESETS.some(([, v]) => v === cfg.family)}, 'Custom…'),
  );
  const famInput = el('input', {cls: 'uk-input craft-font-ctrl', type: 'text', value: cfg.family, placeholder: 'CSS font-family stack'});
  on(famSel, 'change', () => {
    if (famSel.value !== '__custom__') { cfg.family = famSel.value; famInput.value = cfg.family; updatePreview(); }
  });
  on(famInput, 'input', () => {
    cfg.family = famInput.value.trim() || t.defFamily;
    famSel.value = FAMILY_PRESETS.some(([, v]) => v === cfg.family) ? cfg.family : '__custom__';
    updatePreview();
  });

  const sizeVal = el('span', {cls: 'craft-font-val'}, `${cfg.size}px`);
  const sizeInput = el('input', {type: 'range', min: '8', max: '24', step: '0.5', value: String(cfg.size), cls: 'craft-font-range'});
  on(sizeInput, 'input', () => { cfg.size = parseFloat(sizeInput.value); sizeVal.textContent = `${cfg.size}px`; updatePreview(); });

  const lhVal = el('span', {cls: 'craft-font-val'}, String(cfg.lh));
  const lhInput = el('input', {type: 'range', min: '1.0', max: '2.0', step: '0.05', value: String(cfg.lh), cls: 'craft-font-range'});
  on(lhInput, 'input', () => { cfg.lh = parseFloat(lhInput.value); lhVal.textContent = String(cfg.lh); updatePreview(); });

  const wtVal = el('span', {cls: 'craft-font-val'}, String(cfg.weight));
  const wtInput = el('input', {type: 'range', min: '200', max: '700', step: '20', value: String(cfg.weight), cls: 'craft-font-range'});
  on(wtInput, 'input', () => { cfg.weight = parseInt(wtInput.value, 10); wtVal.textContent = String(cfg.weight); updatePreview(); });

  modal.body.replaceChildren(
    el('div', {cls: 'craft-font-grid'},
      el('label', {cls: 'craft-font-row'}, el('span', {}, 'Family preset'), famSel),
      el('label', {cls: 'craft-font-row'}, el('span', {}, 'CSS font-family'), famInput),
      el('label', {cls: 'craft-font-row'}, el('span', {}, 'Size ', sizeVal), sizeInput),
      el('label', {cls: 'craft-font-row'}, el('span', {}, 'Line height ', lhVal), lhInput),
      el('label', {cls: 'craft-font-row'}, el('span', {}, 'Weight ', wtVal), wtInput),
    ),
    el('div', {cls: 'craft-font-prev-wrap'},
      el('div', {cls: 'craft-font-prev-lbl'}, `Live preview — ${t.label}`),
      preview,
    ),
  );
}

let revertOnClose = true;

function openModal() {
  if (!modal) {
    modal = mountModal('font_modal', {
      refs: true,
      signal: ac.signal,
      onClose: () => {
        if (revertOnClose) { current = loadSettings(); applySettings(current); }
        revertOnClose = true;
      },
    });
    on(modal.save, 'click', () => {
      store.setJson(STORE_KEY, current);
      applySettings(current);
      revertOnClose = false;
      modal.close();
      notify('Font settings saved.', 'success');
    }, sig);
    on(modal.reset, 'click', () => {
      current = defaults();
      store.del(STORE_KEY);
      applySettings(current);
      renderModal();
      notify('Font settings reset to defaults.');
    }, sig);
    on(modal.copyCss, 'click', async () => {
      const css = buildCssSnippet(current);
      try {
        await navigator.clipboard.writeText(css);
        notify('Copied CRAFT.css font snippet to clipboard.', 'success');
      } catch {
        prompt('Copy CRAFT.css snippet:', css);
      }
    }, sig);
  }
  current = loadSettings();
  revertOnClose = true;
  renderModal();
  modal.open();
}

// Apply persisted settings immediately on load
if (store.json(STORE_KEY, null)) applySettings(current);

window.__hotkeyRegister?.({id: 'craft-font-settings', label: 'Font settings (by cell type)', group: 'CRAFT · Appearance', combo: '', handler: () => { openModal(); return true; }});
window.__hotkeyRegister?.({id: 'craft-font-reset', label: 'Reset font settings to defaults', group: 'CRAFT · Appearance', combo: '', handler: () => {
  current = defaults(); store.del(STORE_KEY); applySettings(current);
  notify('Font settings reset to defaults.');
  return true;
}});

window.craftFonts = {
  open: openModal,
  get: () => loadSettings(),
  apply: applySettings,
  reset: () => { current = defaults(); store.del(STORE_KEY); applySettings(current); },
  snippet: () => buildCssSnippet(current),
  disable() {
    ac.abort();
    modal?.destroy?.();
    delete window.craftFonts;
  },
};

## Font Settings CSS (`#| export_css`)

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Font Settings Modal (14_font_settings) ────────────────────────── */
.craft-font-card { width: min(620px, 92vw); max-height: 80vh; display: flex; flex-direction: column; overflow: hidden; }
.craft-font-main { display: flex; flex-direction: column; overflow-y: auto; }
.craft-font-tabs { display: flex; flex-wrap: wrap; gap: 4px; padding: 10px 14px 0; border-bottom: 1px solid var(--craft-border); }
.craft-font-tab {
  padding: 6px 10px; border: 0; border-bottom: 2px solid transparent; background: transparent;
  color: var(--craft-muted); font-size: 11.5px; cursor: pointer;
}
.craft-font-tab:hover { color: var(--craft-fg-strong); }
.craft-font-tab.is-active { color: var(--craft-accent); border-bottom-color: var(--craft-accent); font-weight: 600; }
.craft-font-body { padding: 14px 16px; display: flex; flex-direction: column; gap: 14px; }
.craft-font-grid { display: flex; flex-direction: column; gap: 10px; }
.craft-font-row { display: grid; grid-template-columns: 130px 1fr; align-items: center; gap: 10px; font-size: 11.5px; color: var(--craft-fg); }
.craft-font-val { font-family: var(--craft-code-family); color: var(--craft-accent); margin-left: 4px; }
.craft-font-ctrl { height: 28px !important; font-size: 11.5px !important; }
.craft-font-range { width: 100%; accent-color: var(--craft-accent); }
.craft-font-prev-wrap { padding: 10px 12px; border-radius: 6px; background: var(--craft-bg); border: 1px solid var(--craft-border); }
.craft-font-prev-lbl { font-size: 10px; text-transform: uppercase; letter-spacing: .04em; color: var(--craft-meta-fg, var(--craft-muted)); margin-bottom: 6px; }
.craft-font-preview { color: var(--craft-fg); white-space: pre-wrap; }
.craft-font-preview.is-code, .craft-font-preview.is-raw { background: var(--craft-recessed); padding: 8px 10px; border-radius: 4px; }

Verify `14_font_settings.ipynb` exports `window.craftFonts`, its template, and CSS.

In [ ]:
import os, subprocess
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/14_font_settings.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
css = '\n'.join(code for _, code in assets['CRAFT.css'])
for sym in ('window.craftFonts', 'font_modal', 'buildCssSnippet', 'syncMonaco'):
    assert sym in js, f"Missing {sym}"
assert '.craft-font-card' in css and '.craft-font-preview' in css

node_script = r'''
const assert = require('assert');
const src = process.env.FONT_JS;
const i0 = src.indexOf('const TYPES =');
const i1 = src.indexOf('const loadSettings =');
const j0 = src.indexOf('function buildCssSnippet');
const j1 = src.indexOf('function renderModal');
const {defaults, buildCssSnippet} = new Function(
  src.slice(i0, i1) + '\n' + src.slice(j0, j1) + '\nreturn {defaults, buildCssSnippet};'
)();
const cfg = defaults();
const out = buildCssSnippet(cfg);
assert(out.startsWith(':root {') && out.endsWith('}'));
for (const k of ['prompt', 'response', 'note', 'code', 'raw']) {
  assert(out.includes(`--craft-font-${k}-family:`), `missing ${k} family`);
  assert(out.includes(`--craft-font-${k}-size:`), `missing ${k} size`);
}
'''
subprocess.run(['node', '-e', node_script], env={**os.environ, 'FONT_JS': js}, check=True)
